In [ ]:
import tensorflow as tf
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
import json
base_path = "/content/drive/MyDrive/plant_disease_project"


from google.colab import drive
drive.mount('/content/drive')


val_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    validation_split=0.2
)
train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    validation_split=0.2,
    rotation_range=25,
    zoom_range=0.2,
    shear_range=0.2,
    horizontal_flip=True,
    brightness_range=[0.7, 1.3],
    width_shift_range=0.15,
    height_shift_range=0.15,
    fill_mode='nearest'
)

train_data = train_datagen.flow_from_directory(
    f"{base_path}/dataset/final_dataset",
    target_size=(160, 160),
    batch_size=32,
    class_mode='categorical',
    subset='training',
    shuffle=True
)
val_data = val_datagen.flow_from_directory(
    f"{base_path}/dataset/final_dataset",
    target_size=(160, 160),
    batch_size=32,
    class_mode='categorical',
    subset='validation',
    shuffle=False
)
print("✅ Data loaded")


model = tf.keras.models.load_model(
    f"{base_path}/models/best_extraction_model.keras"
)
print("✅ Model loaded")
model.summary()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Found 16516 images belonging to 15 classes.
Found 4122 images belonging to 15 classes.
✅ Data loaded
✅ Model loaded


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ mobilenetv2_1.00_160            │ (None, 5, 5, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_1      │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 1280)           │         5,120 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 256)            │       327,936 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 15)             │         1,935 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,356,527 (12.80 MB)

 Trainable params: 365,327 (1.39 MB)

 Non-trainable params: 2,260,544 (8.62 MB)

 Optimizer params: 730,656 (2.79 MB)

In [ ]:
base_model = model.layers[0]
base_model.trainable = True
for layer in base_model.layers[:-30]:
    layer.trainable = False


model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)


callbacks = [
    EarlyStopping(monitor='val_loss', patience=5,
                  restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.3,
                      patience=3, min_lr=1e-7, verbose=1),
    ModelCheckpoint(f"{base_path}/models/best_finetuned_model.keras",
                    monitor='val_accuracy', save_best_only=True, verbose=1)
]

history_fine = model.fit(
    train_data,
    validation_data=val_data,
    epochs=10,
    callbacks=callbacks
)
print("✅ Fine tuning complete")

Epoch 1/10
517/517 ━━━━━━━━━━━━━━━━━━━━ 0s 10s/step - accuracy: 0.8261 - loss: 0.5568 
Epoch 1: val_accuracy improved from None to 0.87239, saving model to /content/drive/MyDrive/plant_disease_project/models/best_finetuned_model.keras

Epoch 1: finished saving model to /content/drive/MyDrive/plant_disease_project/models/best_finetuned_model.keras
517/517 ━━━━━━━━━━━━━━━━━━━━ 6634s 13s/step - accuracy: 0.8403 - loss: 0.5054 - val_accuracy: 0.8724 - val_loss: 0.4731 - learning_rate: 1.0000e-05
Epoch 2/10
517/517 ━━━━━━━━━━━━━━━━━━━━ 0s 353ms/step - accuracy: 0.8655 - loss: 0.4223
Epoch 2: val_accuracy improved from 0.87239 to 0.89738, saving model to /content/drive/MyDrive/plant_disease_project/models/best_finetuned_model.keras

Epoch 2: finished saving model to /content/drive/MyDrive/plant_disease_project/models/best_finetuned_model.keras
517/517 ━━━━━━━━━━━━━━━━━━━━ 199s 384ms/step - accuracy: 0.8727 - loss: 0.4049 - val_accuracy: 0.8974 - val_loss: 0.3348 - learning_rate: 1.0000e-05
E

In [ ]:
history_fine2 = model.fit(
    train_data,
    validation_data=val_data,
    epochs=15,           # 15 more epochs - EarlyStopping will cut short when done
    callbacks=callbacks  # same callbacks already defined
)

Epoch 1/15
517/517 ━━━━━━━━━━━━━━━━━━━━ 0s 366ms/step - accuracy: 0.9282 - loss: 0.2168
Epoch 1: val_accuracy did not improve from 0.93571
517/517 ━━━━━━━━━━━━━━━━━━━━ 206s 398ms/step - accuracy: 0.9271 - loss: 0.2214 - val_accuracy: 0.9355 - val_loss: 0.1975 - learning_rate: 1.0000e-05
Epoch 2/15
517/517 ━━━━━━━━━━━━━━━━━━━━ 0s 369ms/step - accuracy: 0.9310 - loss: 0.2033
Epoch 2: val_accuracy improved from 0.93571 to 0.94153, saving model to /content/drive/MyDrive/plant_disease_project/models/best_finetuned_model.keras

Epoch 2: finished saving model to /content/drive/MyDrive/plant_disease_project/models/best_finetuned_model.keras
517/517 ━━━━━━━━━━━━━━━━━━━━ 208s 402ms/step - accuracy: 0.9285 - loss: 0.2162 - val_accuracy: 0.9415 - val_loss: 0.1836 - learning_rate: 1.0000e-05
Epoch 3/15
517/517 ━━━━━━━━━━━━━━━━━━━━ 0s 366ms/step - accuracy: 0.9316 - loss: 0.2091
Epoch 3: val_accuracy did not improve from 0.94153
517/517 ━━━━━━━━━━━━━━━━━━━━ 206s 397ms/step - accuracy: 0.9307 - loss:

In [ ]:
import tensorflow as tf
import json

# 1. Load best finetuned model
model = tf.keras.models.load_model(
    f"{base_path}/models/best_finetuned_model.keras"
)
print("✅ Model loaded")

# 2. Convert to TFLite
converter = tf.lite.TFLiteConverter.from_keras_model(model)

converter.optimizations = []
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS]

tflite_model = converter.convert()
print("✅ Model converted")

# 3. Save TFLite model
tflite_path = f"{base_path}/models/plant_model.tflite"
with open(tflite_path, "wb") as f:
    f.write(tflite_model)
print(f"✅ TFLite model saved")

# 4. Check file size
import os
size = os.path.getsize(tflite_path) / (1024*1024)
print(f"📦 Model size: {size:.2f} MB")

✅ Model loaded
Saved artifact at '/tmp/tmppgpi6d29'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 160, 160, 3), dtype=tf.float32, name='input_layer_2')
Output Type:
  TensorSpec(shape=(None, 15), dtype=tf.float32, name=None)
Captures:
  134813117516688: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134812704855760: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134812704850384: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134812704855952: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134812704856720: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134812704850960: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134812704851152: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134812704856528: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134812704855568: TensorSpec(shape=(), dtype=tf.resource, name=None)
  134812704857680: TensorSpec(shape=(), dtype=tf.resource, name=None)

In [ ]:
interpreter = tf.lite.Interpreter(model_path=tflite_path)
interpreter.allocate_tensors()

input_details  = interpreter.get_input_details()
output_details = interpreter.get_output_details()

print("✅ TFLite model verified")
print(f"Input shape:  {input_details[0]['shape']}")   # should be [1,160,160,3]
print(f"Input dtype:  {input_details[0]['dtype']}")   # should be float32
print(f"Output shape: {output_details[0]['shape']}")  # should be [1,15]
print(f"Output dtype: {output_details[0]['dtype']}")  # should be float32

✅ TFLite model verified
Input shape:  [  1 160 160   3]
Input dtype:  <class 'numpy.float32'>
Output shape: [ 1 15]
Output dtype: <class 'numpy.float32'>


/usr/local/lib/python3.12/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


In [ ]:
class_names = list(train_data.class_indices.keys())
with open(f"{base_path}/class_names.json", "w") as f:
    json.dump(class_names, f)
print("✅ Class names saved")
print(f"Classes: {class_names}")

✅ Class names saved
Classes: ['Pepper bell___Bacterial spot', 'Pepper bell___healthy', 'Potato___Early blight', 'Potato___Late blight', 'Potato___healthy', 'Tomato___Bacterial spot', 'Tomato___Early blight', 'Tomato___Late blight', 'Tomato___Leaf Mold', 'Tomato___Septoria leaf spot', 'Tomato___Spider mites Two spotted spider mite', 'Tomato___Target Spot', 'Tomato___Tomato YellowLeaf Curl Virus', 'Tomato___Tomato mosaic virus', 'Tomato___healthy']
